# Inference Production Runner

Run `inference_production` using the trained checkpoint and input parquet files in `data_v2`.

In [ ]:
from pathlib import Path
import torch

# Paths relative to the repository root; adapt to your local artifacts.
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
model_dir = repo_root / 'data' / 'artifacts' / 'model_tgcn_2026_01_27'
data_v2 = repo_root / 'data_v2'
base_repo = repo_root

model_path = model_dir / 'model.pth'

# Data inputs (parquets)
jams_path = data_v2 / 'df_jams_test.parquet'
irregularities_path = data_v2 / 'df_irregularidades_test.parquet'
weather_path = data_v2 / 'df_clima_test.parquet'
pmt_puntos_path = data_v2 / 'df_pmt_puntos_test.parquet'
pmt_tramos_path = data_v2 / 'df_pmt_tramos_test.parquet'
events_path = data_v2 / 'df_eventos_test.parquet'
incidents_path = data_v2 / "df_incidentes_test.parquet"

# Shared assets (graph/static/norm) from training preprocess
processed_data_dir = base_repo / 'data' / 'datos_procesados'
graph_path = processed_data_dir / 'conectivity_graph' / 'road_graph.pickle'
static_path = processed_data_dir / 'sparse_datasets' / 'static_dataset.h5'
norm_path = processed_data_dir / 'sparse_datasets' / 'normalization_constants.pkl'

# Optional output path
output_path = base_repo / 'data' / 'artifacts' / 'inference_preds.npy'

for p in [model_path, jams_path, irregularities_path, weather_path, pmt_puntos_path, pmt_tramos_path, events_path, graph_path, static_path, norm_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('Paths OK')


In [ ]:
model_path

In [ ]:
import sys
import pickle
import torch
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.gnn_temporal import GNN_Temporal_Model

# --- Load checkpoint ---
checkpoint = torch.load(model_path, weights_only=False)
model_kwargs = checkpoint.get('model_kwargs', {})
keys_to_drop = {"include_weather", "num_weather_features"}
for k in keys_to_drop:
    model_kwargs.pop(k, None)
state_dict = checkpoint.get('state_dict', None)

if state_dict is None:
    raise ValueError('Checkpoint missing state_dict. Cannot load model weights.')

# Build model
model = GNN_Temporal_Model(
    edge_index=None,
    **model_kwargs,
)
model.load_state_dict(state_dict)

# Move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print('Model loaded:', model.__class__.__name__, 'device=', device)
print('model_kwargs:', model_kwargs)


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    repo_root = cwd.parent
else:
    repo_root = cwd

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from src.inference_production import InferencePipeline

# Initialize pipeline without auto-loading model
pipeline = InferencePipeline(
    graph_path=str(graph_path),
    static_data_path=str(static_path),
    normalization_constants_path=str(norm_path),
    model_path=None,
    use_ray=False,
    expected_feature_map_path=  str(repo_root / 'data' / 'artifacts' / 'feature_maps' / 'expected_feature_map.by_dataset.json'),
)

# Inject the model we loaded from checkpoint
pipeline.model = model

results = pipeline.predict(
    jams_path=str(jams_path),
    irregularities_path=str(irregularities_path),
    weather_path=str(weather_path),
    pmt_puntos_path=str(pmt_puntos_path),
    pmt_tramos_path=str(pmt_tramos_path),
    events_path=str(events_path),
    incidents_path=str(incidents_path),
    output_path=str(output_path),
)

print('Results keys:', results.keys())
print('num_batches:', results['num_batches'])
print('num_nodes:', results['num_nodes'])
print('load_time:', results['load_time'])
print('inference_time:', results['inference_time'])
print('total_time:', results['total_time'])

if results['predictions'] is not None:
    pred = results['predictions'][0]
    print('First prediction shape:', pred.shape)


In [ ]:
print(model)

In [ ]:
pred

In [ ]:
# --- Full-graph inference using seed list + hop filtering ---
from pathlib import Path
import json

seed_path = (repo_root / 'data' / 'artifacts') / "seed_lists" / "seeds_k6.json"
seed_payload = json.loads(seed_path.read_text())
seed_list = seed_payload.get("seeds", seed_payload)

results = pipeline.predict(
    jams_path=str(jams_path),
    irregularities_path=str(irregularities_path),
    weather_path=str(weather_path),
    pmt_puntos_path=str(pmt_puntos_path),
    pmt_tramos_path=str(pmt_tramos_path),
    events_path=str(events_path),
    incidents_path=str(incidents_path),
    node_seeds=seed_list,
    max_hops=6,
)

pred_df = results.get("predictions_df")
print(pred_df.head())
print("rows:", len(pred_df))